In [1]:
import logging

# Add this line to suppress the nibabel warnings
logging.getLogger('nibabel').setLevel(logging.ERROR)

In [1]:
!pip install segmentation_models_pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninsta

In [3]:
import logging
import os
import random
import re
import time
from pathlib import Path
import pathlib         
from typing import Dict, Tuple
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from scipy.ndimage import (
    binary_closing,
    generate_binary_structure,
    label,
    zoom,
)
from skimage.morphology import (
    binary_opening,
    disk,
    remove_small_holes,
    remove_small_objects,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset, Subset
import warnings 
import segmentation_models_pytorch as smp
import torch.nn.functional as F
from tqdm.auto import tqdm 
from torch.optim import Adam
from skimage.morphology import remove_small_objects, remove_small_holes, disk, ball
from scipy.ndimage    import label, binary_closing


In [4]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=3, init_features=32):
        super().__init__()
        feats = init_features

        # Encoder
        self.enc1 = UNet._block(in_channels,    feats,       name="enc1")
        self.pool = nn.MaxPool2d(2,2)
        self.enc2 = UNet._block(feats,          feats*2,     name="enc2")
        self.enc3 = UNet._block(feats*2,        feats*4,     name="enc3")
        self.enc4 = UNet._block(feats*4,        feats*8,     name="enc4")

        # Bottleneck
        self.bottleneck = UNet._block(feats*8, feats*16,    name="bottleneck")

        # Decoder
        self.up4 = nn.ConvTranspose2d(feats*16, feats*8, 2,2)
        self.dec4 = UNet._block(feats*16,      feats*8,     name="dec4")
        self.up3 = nn.ConvTranspose2d(feats*8,  feats*4, 2,2)
        self.dec3 = UNet._block(feats*8,       feats*4,     name="dec3")
        self.up2 = nn.ConvTranspose2d(feats*4,  feats*2, 2,2)
        self.dec2 = UNet._block(feats*4,       feats*2,     name="dec2")
        self.up1 = nn.ConvTranspose2d(feats*2,  feats,   2,2)
        self.dec1 = UNet._block(feats*2,       feats,       name="dec1")

        # Final 1×1 conv → 3 channels (BG, liver, tumor)
        self.conv = nn.Conv2d(feats, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder pass
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder pass + skip connections
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # raw logits → shape (B,3,H,W)
        return self.conv(d1)

    @staticmethod
    def _block(in_ch, features, name):
        return nn.Sequential(
            nn.Conv2d(in_ch,    features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True),
            nn.Conv2d(features, features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True),
        )



In [5]:
class UNetPlusPlus(nn.Module):
    def __init__(self, in_channels=1, out_channels=3, init_features=32):
        super().__init__()
        feats = init_features

        # ---- Encoder ----
        self.enc1 = self._block(in_channels,       feats,    name="enc1")
        self.enc2 = self._block(feats,             feats*2,  name="enc2")
        self.enc3 = self._block(feats*2,           feats*4,  name="enc3")
        self.enc4 = self._block(feats*4,           feats*8,  name="enc4")
        self.pool = nn.MaxPool2d(2,2)

        # ---- Bottleneck ----
        self.bottleneck = self._block(feats*8, feats*16, name="bottleneck")

        # ---- Decoder ----
        self.up4 = nn.ConvTranspose2d(feats*16, feats*8, 2,2)
        self.dec4 = self._block(feats*8*2,  feats*8,  name="dec4")

        self.up3 = nn.ConvTranspose2d(feats*8,  feats*4, 2,2)
        self.dec3 = self._block(feats*4*2,  feats*4,  name="dec3")

        self.up2 = nn.ConvTranspose2d(feats*4,  feats*2, 2,2)
        self.dec2 = self._block(feats*2*2,  feats*2,  name="dec2")

        self.up1 = nn.ConvTranspose2d(feats*2,  feats,    2,2)
        self.dec1 = self._block(feats*1*2,  feats,    name="dec1")

        # ---- Final 1×1 conv to 3 classes ----
        self.classifier = nn.Conv2d(feats, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b  = self.bottleneck(self.pool(e4))

        # Decoder with skip-connections
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # raw logits: shape (B,3,H,W)
        return self.classifier(d1)

    @staticmethod
    def _block(in_ch, out_ch, name=None):
        return nn.Sequential(
            nn.Conv2d(in_ch,   out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch,  out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


In [6]:
base1 = Path('/kaggle/input/liver-tumor-segmentation')
base2 = Path('/kaggle/input/liver-tumor-segmentation-part-2')

# grab all .nii files under both roots
all_files = list(base1.rglob('*.nii')) + list(base2.rglob('*.nii'))
records = []
pattern = re.compile(r'(?P<type>volume|segmentation)-(?P<id>\d+)\.nii$')

for p in all_files:
    m = pattern.search(p.name)
    if not m: 
        continue
    rec = {
        'id'       : int(m.group('id')),
        'type'     : m.group('type'),
        'filepath' : str(p)
    }
    records.append(rec)

df = pd.DataFrame(records)

# pivot so each row is one case, with two columns: ct and mask
df_wide = df.pivot(index='id', columns='type', values='filepath').reset_index()
df_wide.columns.name = None
df_wide = df_wide.rename(columns={'volume':'ct_path','segmentation':'mask_path'})

df_wide.head()
df_small = df_wide[:50]

In [7]:
df_train, df_tmp  = train_test_split(df_small, test_size=0.4, random_state=42)
df_val,   df_test = train_test_split(df_tmp,   test_size=0.5, random_state=42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ----------------------------------------------
# 1. Augmentations
# ----------------------------------------------

base_tx = A.Compose([
    A.Resize(256,256), A.Normalize(0,1), ToTensorV2()
])
aug_tx  = A.Compose([
    A.HorizontalFlip(0.5), A.VerticalFlip(0.5), A.RandomRotate90(0.5),
    A.Affine(translate_percent=0.05, scale=(0.9,1.1), rotate=(-15,15), p=0.5),
    A.ElasticTransform(alpha=1,sigma=50,p=0.2),
    A.RandomBrightnessContrast(0.1,0.1,p=0.5),
    A.Resize(256,256), A.Normalize(0,1), ToTensorV2(),
])

stage1_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(translate_percent=0.1, scale=(0.9,1.1), rotate=(-20,20), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.Resize(256,256),
    A.Normalize(mean=0.0, std=1.0),
    ToTensorV2(),
])

stage1_val_transform = A.Compose([
    A.Resize(256,256),
    A.Normalize(mean=0.0, std=1.0),
    ToTensorV2(),
])
stage2_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(translate_percent=0.05, scale=(0.9,1.1), rotate=(-15,15), p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
    A.Resize(256,256),
    A.Normalize(mean=0.0, std=1.0),
    ToTensorV2(),
])
stage2_val_transform = A.Compose([
    A.Resize(256,256),
    A.Normalize(mean=0.0, std=1.0),
    ToTensorV2(),
])

In [10]:
def extract_liver_roi(mask2d: np.ndarray, pad: int=10):
    ys, xs = np.nonzero(mask2d)
    if len(ys)==0:
        return 0, mask2d.shape[0], 0, mask2d.shape[1]
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    y0, y1 = max(0, y0-pad), min(mask2d.shape[0], y1+pad)
    x0, x1 = max(0, x0-pad), min(mask2d.shape[1], x1+pad)
    return y0, y1, x0, x1

In [12]:
def train_validate(model, train_loader, val_loader,
                   criterion, optimizer, scheduler,
                   num_epochs=25, patience=5,
                   device=torch.device('cpu'),
                   ckpt_path='best.pt'):
    best_val, wait = float('inf'), 0
    hist = []
    for epoch in range(1, num_epochs+1):
        # train
        model.train(); tloss = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device).long()
            outs = model(imgs)
            loss = criterion(outs, masks)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tloss += loss.item()
        tloss /= len(train_loader)
        # val
        model.eval(); vloss = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device).long()
                vloss += criterion(model(imgs), masks).item()
        vloss /= len(val_loader)
        # scheduler & save
        scheduler.step(vloss)
        hist.append([epoch, tloss, vloss])
        print(f"[Epoch {epoch:02}] Train: {tloss:.4f} | Val: {vloss:.4f}")
        if vloss < best_val - 1e-4:
            best_val, wait = vloss, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break
    return np.array(hist)


In [8]:

def resample_slice(slice2d: np.ndarray,
                   in_spacing: Tuple[float,float],
                   out_spacing: Tuple[float,float],
                   is_mask: bool=False) -> np.ndarray:
    zoom_y = in_spacing[0] / out_spacing[0]
    zoom_x = in_spacing[1] / out_spacing[1]
    new_h  = int(round(slice2d.shape[0] * zoom_y))
    new_w  = int(round(slice2d.shape[1] * zoom_x))
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    return cv2.resize(slice2d, (new_w, new_h), interpolation=interp)

class LiTSDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        width: float,
        level: float,
        target_size: Tuple[int,int] = (256, 256),
        transform=None
    ):
        super().__init__()
        self.width, self.level = width, level
        self.out_h, self.out_w = target_size
        self.transform = transform

        # build list of only those slices that have class=1 or class=2
        self.samples = []
        for _, row in df.iterrows():
            ct_f, msk_f = row["ct_path"], row["mask_path"]
            mask_vol = nib.load(msk_f).get_fdata()  # shape (H,W,D)
            # find z-indices with any liver(1) or tumour(2)
            fg_slices = np.where(
                np.any((mask_vol==1)|(mask_vol==2), axis=(0,1))
            )[0]
            for z in fg_slices:
                self.samples.append((ct_f, msk_f, int(z)))

        if not self.samples:
            raise RuntimeError("No foreground slices found!")

        # cache for nibabel objects
        self._cache = {}

    @staticmethod
    def window_norm(img: np.ndarray, width: float, level: float) -> np.ndarray:
        lo, hi = level - width/2, level + width/2
        img = np.clip(img, lo, hi)
        return (img - lo) / (hi - lo + 1e-5)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        ct_f, msk_f, z = self.samples[idx]

        # lazy‐load volumes
        if ct_f not in self._cache:
            self._cache[ct_f] = nib.load(ct_f)
        if msk_f not in self._cache:
            self._cache[msk_f] = nib.load(msk_f)

        ct_vol  = self._cache[ct_f]
        msk_vol = self._cache[msk_f]

        # extract slice
        ct_slice   = ct_vol.dataobj[..., z].astype(np.float32)
        mask_slice = msk_vol.dataobj[..., z].astype(np.uint8)

        # resample to 1 mm in‐plane
        dy, dx     = ct_vol.header.get_zooms()[:2]
        ct_slice   = resample_slice(ct_slice,   (dy,dx), (1.0,1.0), is_mask=False)
        mask_slice = resample_slice(mask_slice, (dy,dx), (1.0,1.0), is_mask=True)

        # window & normalize
        img = self.window_norm(ct_slice, self.width, self.level).astype(np.float32)

        # morphological cleanup (optional steps commented out)
        cleaned = np.zeros_like(mask_slice, dtype=np.uint8)
        selem   = disk(3)
        for cls in (1,2):
            region = (mask_slice == cls)
            # e.g. region = remove_small_objects(region, min_size=100)
            # region = binary_opening(region, selem)
            # region = binary_closing(region, selem)
            # region = remove_small_holes(region, area_threshold=500)
            cleaned[region] = cls
        mask = cleaned

        # resize to network input
        img  = cv2.resize(img,  (self.out_w, self.out_h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.out_w, self.out_h), interpolation=cv2.INTER_NEAREST)

        if self.transform:
            sample      = self.transform(image=img, mask=mask)
            img_tensor  = sample["image"]
            mask_tensor = sample["mask"].long()
        else:
            img_tensor  = torch.from_numpy(img).unsqueeze(0).float()
            mask_tensor = torch.from_numpy(mask).long()

        return img_tensor, mask_tensor


In [9]:
weights = torch.tensor([0.05, 1.0, 3.0], device=device)
ce   = nn.CrossEntropyLoss(weight=weights)
dice = smp.losses.DiceLoss(mode="multiclass", from_logits=True)
criterion = lambda out, y: 0.5*ce(out, y) + 0.5*dice(out, y)

In [10]:

def post_process(
    pred: np.ndarray,
    min_liver: int = 500,
    min_tumour: int = 30,
    merge_dist: int = 2
) -> np.ndarray:
    """
    2-D post-processing of a single slice (H, W) of {0,1,2} labels.
    """
    cleaned = np.zeros_like(pred, dtype=np.uint8)

    # ---------- Liver (class=1) ----------
    liver = (pred == 1)
    liver = remove_small_objects(liver, min_size=min_liver, connectivity=2)
    if liver.any():
        lbl, _   = label(liver, structure=np.ones((3,3), dtype=bool))
        areas    = np.bincount(lbl.ravel())
        areas[0] = 0
        liver    = (lbl == areas.argmax())
    cleaned[liver] = 1

    # ---------- Tumour (class=2) ----------
    tumour = (pred == 2)
    if merge_dist > 0:
        # use a 2-D disk, not a 3-D ball
        selem   = disk(merge_dist)
        tumour  = binary_closing(tumour, structure=selem)
    tumour = remove_small_objects(tumour, min_size=min_tumour, connectivity=2)
    tumour = remove_small_holes(tumour, area_threshold=min_tumour)
    cleaned[tumour] = 2

    return cleaned


In [12]:
def train_single(model, name):
    print(f"\n=== Single-stage {name} ===")
    # Data loaders
    train_loader = DataLoader(
        LiTSDataset(df_train, width=350, level=40, target_size=(256,256), transform=aug_tx),
        batch_size=8, shuffle=True,  num_workers=4
    )
    val_loader = DataLoader(
        LiTSDataset(df_val,   width=350, level=40, target_size=(256,256), transform=base_tx),
        batch_size=8, shuffle=False, num_workers=4
    )

    optimizer = Adam(model.parameters(), lr=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3)

    best_val  = float('inf')
    best_path = None

    for epoch in range(1, 11):
        print(f"[Epoch {epoch} {name}]")

        # — Training —
        model.train()
        train_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outs  = model(imgs)
            loss  = criterion(outs, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # — Validation & collect predictions and CT slices —
        model.eval()
        val_loss = 0.0
        raw_preds, raw_trues, raw_imgs = [], [], []
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                outs = model(imgs)
                val_loss += criterion(outs, masks).item()

                preds = outs.argmax(1).cpu().numpy()  # shape (B,H,W)
                gts   = masks.cpu().numpy()
                cts   = imgs.cpu().numpy()             # shape (B,1,H,W)

                for i in range(preds.shape[0]):
                    raw_preds.append(preds[i])
                    raw_trues.append(gts[i])
                    raw_imgs.append(cts[i,0])  # drop channel dim

        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        print(f"  train {train_loss:.4f} | val {val_loss:.4f}")

        # — Save best model —
        if val_loss < best_val - 1e-4:
            best_val  = val_loss
            best_path = f"best_{name}.pt"
            torch.save(model.state_dict(), best_path)
            print(f"  → Saved best weights to {best_path}")

        # — Compute confusion matrix diagonals —
        all_raw = np.concatenate([r.ravel() for r in raw_preds])
        all_gt  = np.concatenate([t.ravel() for t in raw_trues])
        all_pp  = np.concatenate([post_process(r).ravel() for r in raw_preds])

        cm_raw = confusion_matrix(all_gt, all_raw, labels=[0,1,2])
        cm_pp  = confusion_matrix(all_gt, all_pp,  labels=[0,1,2])
        print("  raw diag:", np.diag(cm_raw), "| pp diag:", np.diag(cm_pp))

        # — Plot 3 random validation examples (CT / raw / post-proc / diff) —
        picks = random.sample(range(len(raw_preds)), k=3)
        for idx in picks:
            ct  = raw_imgs[idx]
            rp  = raw_preds[idx]
            pp  = post_process(rp)
            diff = (rp != pp)

            fig, axes = plt.subplots(1, 4, figsize=(12, 3))
            axes[0].imshow(ct,  cmap='gray'); axes[0].set_title('CT slice')
            axes[1].imshow(rp,  cmap='gray'); axes[1].set_title('Raw pred')
            axes[2].imshow(pp,  cmap='gray'); axes[2].set_title('Post-proc')
            axes[3].imshow(diff, cmap='gray'); axes[3].set_title('Difference')
            for ax in axes:
                ax.axis('off')
            plt.suptitle(f'{name} Epoch {epoch}')
            plt.tight_layout()
            plt.show()

    return best_path

# ----------------------------------------------
# 7. Test-set evaluation
# ----------------------------------------------

def evaluate_test_with_postproc(model, name, best_path, df_test, batch_size=8):
    """
    Evaluate a trained model on the test set and plot confusion 
    matrices before and after postprocessing.
    """
    # 1) Test loader
    test_ds = LiTSDataset(
        df_test,
        width=350,
        level=40,
        target_size=(256,256),
        transform=base_tx
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
    )

    # 2) Load best weights
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.to(device).eval()

    # 3) Inference: collect raw preds and ground truths
    raw_preds, gts = [], []
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs = imgs.to(device)
            out  = model(imgs)
            pr   = out.argmax(1).cpu().numpy()
            for p, g in zip(pr, masks.numpy()):
                raw_preds.append(p)
                gts.append(g)

    # 4) Flatten arrays
    raw_flat = np.concatenate([p.ravel() for p in raw_preds])
    gt_flat  = np.concatenate([g.ravel() for g in gts])

    # 5) Postprocess each slice and flatten
    pp_flat  = np.concatenate([post_process(p).ravel() for p in raw_preds])

    # 6) Compute confusion matrices
    labels = [0,1,2]
    class_names = ['Background','Liver','Tumour']
    cm_raw = confusion_matrix(gt_flat, raw_flat, labels=labels)
    cm_pp  = confusion_matrix(gt_flat, pp_flat,  labels=labels)

    # 7) Plot side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    disp_raw = ConfusionMatrixDisplay(cm_raw, display_labels=class_names)
    disp_pp  = ConfusionMatrixDisplay(cm_pp,  display_labels=class_names)

    disp_raw.plot(ax=axes[0], cmap='Blues', colorbar=False)
    axes[0].set_title(f'{name} Raw CM')

    disp_pp.plot(ax=axes[1],  cmap='Blues', colorbar=False)
    axes[1].set_title(f'{name} Post-processed CM')

    plt.tight_layout()
    plt.show()

    return cm_raw, cm_pp

In [ ]:
# ----------------------------------------------
# 8. Run training & testing for UNet and UNet++
# ----------------------------------------------
unet = UNet().to(device)
best_unet = train_single(unet, "UNet")

unp   = UNetPlusPlus().to(device)
best_unp = train_single(unp, "UNet++")


unet_cm_raw, unet_cm_pp = evaluate_test_with_postproc(
    model=unet,
    name="UNet",
    best_path=best_unet,
    df_test=df_test
)

unp_cm_raw, unp_cm_pp = evaluate_test_with_postproc(
    model=unp,
    name="UNet++",
    best_path=best_unp,
    df_test=df_test
)


=== Single-stage UNet ===


pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1


[Epoch 1 UNet]


In [11]:


class LiverDataset(Dataset):
    """
    Only 2D slices containing any liver (mask==1 or 2).
    Implements caching, resampling, windowing, and optional transform.
    """
    def __init__(
        self,
        df: pd.DataFrame,
        width: float,
        level: float,
        target_size: Tuple[int,int]=(256,256),
        transform=None
    ):
        super().__init__()
        self.width, self.level = width, level
        self.out_h, self.out_w  = target_size
        self.transform         = transform

        # gather only slices with liver present
        self.samples = []
        for _, row in df.iterrows():
            ct_f, ms_f = row['ct_path'], row['mask_path']
            mvol = nib.load(ms_f).get_fdata().astype(np.uint8)
            # find slices where mask==1 or mask==2 appears
            fg = np.any((mvol==1)|(mvol==2), axis=(0,1))
            for z in np.where(fg)[0]:
                self.samples.append((ct_f, ms_f, int(z)))

        if not self.samples:
            raise RuntimeError("No liver-containing slices found!")

        # cache for nibabel objects
        self._cache = {}

    @staticmethod
    def window_norm(img: np.ndarray, width: float, level: float) -> np.ndarray:
        lo, hi = level - width/2, level + width/2
        img = np.clip(img, lo, hi)
        return (img - lo) / (hi - lo + 1e-5)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        ct_f, ms_f, z = self.samples[idx]

        # lazy-load & cache
        if ct_f not in self._cache:
            self._cache[ct_f] = nib.load(ct_f)
        if ms_f not in self._cache:
            self._cache[ms_f] = nib.load(ms_f)
        ct_vol  = self._cache[ct_f]
        ms_vol  = self._cache[ms_f]

        # extract slice
        ct_slice = ct_vol.dataobj[..., z].astype(np.float32)
        m_slice  = ms_vol.dataobj[..., z].astype(np.uint8)

        # resample to 1mm x 1mm
        dy, dx   = ct_vol.header.get_zooms()[:2]
        ct_slice = resample_slice(ct_slice, (dy,dx), (1.0,1.0), is_mask=False)
        m_slice  = resample_slice(m_slice,  (dy,dx), (1.0,1.0), is_mask=True)

        # window & normalize
        img = self.window_norm(ct_slice, self.width, self.level).astype(np.float32)

        # cleanup mask (keep both liver classes)
        mask = (m_slice>0).astype(np.uint8)

        # resize to network input
        img  = cv2.resize(img,  (self.out_w, self.out_h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask,(self.out_w, self.out_h), interpolation=cv2.INTER_NEAREST)

        if self.transform:
            s = self.transform(image=img, mask=mask)
            return s['image'], s['mask'].long()
        else:
            return torch.from_numpy(img).unsqueeze(0), torch.from_numpy(mask).long()



In [12]:
class TumourDataset(Dataset):
    """
    Only 2D slices containing any tumour (mask==2).
    Crops to liver ROI before applying transform.
    """
    def __init__(
        self,
        df: pd.DataFrame,
        width: float,
        level: float,
        target_size: Tuple[int,int]=(256,256),
        transform=None
    ):
        super().__init__()
        self.width, self.level = width, level
        self.out_h, self.out_w  = target_size
        self.transform         = transform

        # gather only slices with tumour present
        self.samples = []
        for _, row in df.iterrows():
            ct_f, ms_f = row['ct_path'], row['mask_path']
            mvol = nib.load(ms_f).get_fdata().astype(np.uint8)
            fg = (mvol==2).any(axis=(0,1))
            for z in np.where(fg)[0]:
                self.samples.append((ct_f, ms_f, int(z)))

        if not self.samples:
            raise RuntimeError("No tumour-containing slices found!")

        self._cache = {}

    @staticmethod
    def window_norm(img: np.ndarray, width: float, level: float) -> np.ndarray:
        lo, hi = level - width/2, level + width/2
        img = np.clip(img, lo, hi)
        return (img - lo) / (hi - lo + 1e-5)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        ct_f, ms_f, z = self.samples[idx]

        # lazy-load & cache
        if ct_f not in self._cache:
            self._cache[ct_f] = nib.load(ct_f)
        if ms_f not in self._cache:
            self._cache[ms_f] = nib.load(ms_f)
        ct_vol = self._cache[ct_f]
        ms_vol = self._cache[ms_f]

        # extract slice
        ct_slice = ct_vol.get_fdata().astype(np.float32)[..., z]
        m_slice  = ms_vol.get_fdata().astype(np.uint8)[..., z]

        # resample to 1mm
        dy, dx   = ct_vol.header.get_zooms()[:2]
        ct_slice = resample_slice(ct_slice, (dy,dx), (1.0,1.0), is_mask=False)
        m_slice  = resample_slice(m_slice,  (dy,dx), (1.0,1.0), is_mask=True)

        # window & normalize
        img = self.window_norm(ct_slice, self.width, self.level).astype(np.float32)

        # get liver bbox from mask>0
        liver_mask = (m_slice > 0).astype(np.uint8)
        ys, xs      = np.nonzero(liver_mask)
        if len(ys)==0:
            # fallback to full slice
            y0,y1,x0,x1 = 0, liver_mask.shape[0], 0, liver_mask.shape[1]
        else:
            y0, y1 = ys.min(), ys.max()
            x0, x1 = xs.min(), xs.max()
            pad = 10
            y0, y1 = max(0, y0-pad), min(liver_mask.shape[0], y1+pad)
            x0, x1 = max(0, x0-pad), min(liver_mask.shape[1], x1+pad)

        roi_img = img [y0:y1, x0:x1]
        roi_msk = (m_slice==2).astype(np.uint8)[y0:y1, x0:x1]

        # resize to network input
        roi_img = cv2.resize(roi_img, (self.out_w, self.out_h), interpolation=cv2.INTER_LINEAR)
        roi_msk = cv2.resize(roi_msk, (self.out_w, self.out_h), interpolation=cv2.INTER_NEAREST)

        if self.transform:
            s = self.transform(image=roi_img, mask=roi_msk)
            return s['image'], s['mask'].long()
        else:
            return torch.from_numpy(roi_img).unsqueeze(0), torch.from_numpy(roi_msk).long()

In [13]:
def evaluate_cascade_test(model_cls, name,
                          best1, best2,
                          width=220, level=100,
                          bs=1, wk=0):
    """
    Memory-efficient version.
    Loads two-stage cascade weights, runs on df_test, then:
      1) Plots 3 confusion matrices.
      2) Shows 4 random test slices.
    """
    # load models
    s1 = model_cls(1,2).to(device)
    s1.load_state_dict(torch.load(best1, map_location=device)); s1.eval()
    s2 = model_cls(1,2).to(device)
    s2.load_state_dict(torch.load(best2, map_location=device)); s2.eval()

    # Ensure pin_memory is False to save a little more memory
    loader = DataLoader(
        LiverDataset(df_test, width, level, transform=stage1_val_transform),
        batch_size=bs, shuffle=False, num_workers=wk, pin_memory=False
    )

    # --- MEMORY FIX: Store flattened 1D arrays, not full 2D images ---
    p1_flat, g1_flat = [], []
    rf_flat, pf_flat, gf_flat = [], [], []
    
    # --- MEMORY FIX: Only store a few full examples for plotting ---
    examples_to_plot = []
    num_examples_to_store = 4

    print("Running evaluation on the test set...")
    with torch.no_grad():
        for img, mask in tqdm(loader, desc=f"Evaluating {name}"):
            img = img.to(device)

            # Stage1 liver vs BG
            out1 = s1(img)
            m1   = out1.argmax(1)[0].cpu().numpy()
            gt1  = (mask[0].cpu().numpy() > 0).astype(np.uint8)
            
            # --- MEMORY FIX: Flatten results immediately ---
            p1_flat.append(m1.ravel())
            g1_flat.append(gt1.ravel())

            # Crop ROI for tumour
            y0, y1, x0, x1 = extract_liver_roi(m1)
            # Handle cases where no liver is detected
            if y1 - y0 <= 0 or x1 - x0 <= 0:
                roi = np.zeros((1, 1)) # Create a dummy small array
            else:
                roi = img[0,0,y0:y1, x0:x1].cpu().numpy()

            # Stage2 tumour vs BG
            inp2 = torch.from_numpy(cv2.resize(roi, (256,256)))\
                           .unsqueeze(0).unsqueeze(0).to(device)
            out2 = s2(inp2)
            m2   = out2.argmax(1)[0].cpu().numpy()
            
            roi_h, roi_w = y1-y0, x1-x0
            m2_up = np.zeros_like(m1, dtype=np.uint8) # Start with an empty mask
            if roi_h > 0 and roi_w > 0:
                # Resize tumour mask back to ROI shape only if ROI is valid
                resized_m2 = cv2.resize(m2.astype(np.uint8), (roi_w, roi_h), interpolation=cv2.INTER_NEAREST)
                m2_up[y0:y1, x0:x1] = resized_m2

            # Assemble full 3-class mask
            full_raw = np.zeros_like(m1, dtype=np.uint8)
            full_raw[m1 == 1] = 1
            full_raw[m2_up == 1] = 2 # Place tumor predictions over the liver

            pp_raw = post_process(full_raw)

            # --- MEMORY FIX: Append flattened arrays ---
            rf_flat.append(full_raw.ravel())
            pf_flat.append(pp_raw.ravel())
            gf_flat.append(mask[0].cpu().numpy().ravel())

            # --- MEMORY FIX: Store only a few full examples ---
            if len(examples_to_plot) < num_examples_to_store:
                img_np = img[0,0].cpu().numpy()
                gt_np = mask[0].cpu().numpy()
                diff = (full_raw != pp_raw)
                examples_to_plot.append((img_np, gt_np, full_raw, pp_raw, diff))

    print("Calculating confusion matrices...")
    # Flatten for confusion matrices (now concatenating lists of 1D arrays)
    p1 = np.concatenate(p1_flat)
    g1 = np.concatenate(g1_flat)
    rf = np.concatenate(rf_flat)
    pf = np.concatenate(pf_flat)
    gf = np.concatenate(gf_flat)

    # Compute confusion matrices
    cm1  = confusion_matrix(g1, p1, labels=[0,1])
    cm_r = confusion_matrix(gf, rf, labels=[0,1,2])
    cm_p = confusion_matrix(gf, pf, labels=[0,1,2])

    # Plot confusion matrices
    fig, axes = plt.subplots(1,3,figsize=(15,5))
    ConfusionMatrixDisplay(cm1, display_labels=['BG','Liver'])\
      .plot(ax=axes[0], cmap='Blues', colorbar=False)
    axes[0].set_title(f'{name} Stage1 CM')
    
    ConfusionMatrixDisplay(cm_r, display_labels=['BG','Liver','Tumour'])\
      .plot(ax=axes[1], cmap='Blues', colorbar=False)
    axes[1].set_title(f'{name} Cascade Raw CM')
    
    ConfusionMatrixDisplay(cm_p, display_labels=['BG','Liver','Tumour'])\
      .plot(ax=axes[2], cmap='Blues', colorbar=False)
    axes[2].set_title(f'{name} Cascade PP CM')
    
    plt.tight_layout(); plt.show()

    # --- MEMORY FIX: Plot using the small list of stored examples ---
    print(f"\n--- Plotting {len(examples_to_plot)} examples ---")
    for ct, gt, rp, ppv, diff in examples_to_plot:
        fig, axes = plt.subplots(1,5,figsize=(20,4))
        axes[0].imshow(ct,  cmap='gray');  axes[0].set_title('CT slice')
        axes[1].imshow(gt,  cmap='gray');  axes[1].set_title('Truth mask')
        axes[2].imshow(rp,  cmap='gray');  axes[2].set_title('Raw pred')
        axes[3].imshow(ppv, cmap='gray');  axes[3].set_title('Post-proc')
        axes[4].imshow(diff,cmap='gray');  axes[4].set_title('Difference')
        for ax in axes: ax.axis('off')
        plt.suptitle(f'{name} Example')
        plt.tight_layout(); plt.show()
    
    return cm1, cm_r, cm_p


In [14]:
# --- per-stage 2-class losses ---
w1    = torch.tensor([0.05, 1.0], device=device)
ce1   = torch.nn.CrossEntropyLoss(weight=w1)
dice1 = smp.losses.DiceLoss(mode="multiclass", from_logits=True)
crit1 = lambda out, y: 0.5*ce1(out, y) + 0.5*dice1(out, y)

w2    = torch.tensor([0.05, 3.0], device=device)
ce2   = torch.nn.CrossEntropyLoss(weight=w2)
dice2 = smp.losses.DiceLoss(mode="multiclass", from_logits=True)
crit2 = lambda out, y: 0.5*ce2(out, y) + 0.5*dice2(out, y)

In [16]:
import os, time, random
import numpy as np
import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# assume crit1, crit2, LiverDataset, TumourDataset,
# stage1_transform, stage1_val_transform, stage2_transform, stage2_val_transform,
# post_process, device, df_train, df_val are all already in scope

def train_cascade(model_cls, name,
                  width=350, level=40,
                  bs=8, wk=0,
                  num_epochs=10, patience=6,
                  ckpt_dir="./checkpoints"):
    """
    Two-stage cascade:
     • Stage1: liver vs BG
     • Stage2: tumour vs BG (inside liver ROI)
    Early-stop on mean F1 (liver+tumour)/2, require +1e-4 to improve.
    Saves:
      <ckpt_dir>/<name>/stage1/best.pt
      <ckpt_dir>/<name>/stage2/best.pt
    Returns:
      (stats_stage1, checkpoint_path_stage1,
       stats_stage2, checkpoint_path_stage2)
    where stats_stageN is a dict containing the final raw/post F1s.
    """
    # create directories
    base = os.path.join(ckpt_dir, name)
    s1_dir = os.path.join(base, "stage1"); os.makedirs(s1_dir, exist_ok=True)
    s2_dir = os.path.join(base, "stage2"); os.makedirs(s2_dir, exist_ok=True)

    # helper to evaluate F1s on a val_loader
    def eval_f1s(val_loader, net, crit):
        net.eval()
        raw_preds, gt_masks = [], []
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs = imgs.to(device)
                outs = net(imgs)
                pr = outs.argmax(1).cpu().numpy()
                raw_preds.extend(pr)
                gt_masks.extend(masks.numpy())
        # flatten
        flat_raw = np.concatenate([p.ravel() for p in raw_preds])
        flat_gt  = np.concatenate([g.ravel() for g in gt_masks])
        cm       = confusion_matrix(flat_gt, flat_raw, labels=[0,1,2])
        # compute F1 per class
        f1_raw = {}
        for cls in (0,1,2):
            TP = cm[cls,cls]
            FP = cm[:,cls].sum() - TP
            FN = cm[cls,:].sum() - TP
            f1_raw[cls] = 2*TP/(2*TP + FP + FN + 1e-8)
        # postprocess slice‐wise and recompute F1 for liver/tumour
        post_preds = [post_process(p) for p in raw_preds]
        flat_pp     = np.concatenate([p.ravel() for p in post_preds])
        cm_pp       = confusion_matrix(flat_gt, flat_pp, labels=[0,1,2])
        f1_pp = {}
        for cls in (0,1,2):
            TP = cm_pp[cls,cls]
            FP = cm_pp[:,cls].sum() - TP
            FN = cm_pp[cls,:].sum() - TP
            f1_pp[cls] = 2*TP/(2*TP + FP + FN + 1e-8)
        return f1_raw, f1_pp

    # ── STAGE 1 ────────────────────────────────────────────────────────────
    train1 = DataLoader(
        LiverDataset(df_train, width, level, transform=stage1_transform),
        batch_size=bs, shuffle=True,  num_workers=wk, pin_memory=False
    )
    val1   = DataLoader(
        LiverDataset(df_val,   width, level, transform=stage1_val_transform),
        batch_size=bs, shuffle=False, num_workers=wk, pin_memory=False
    )

    net1    = model_cls(1,2).to(device)
    opt1    = Adam(net1.parameters(), lr=1e-4)
    sch1    = ReduceLROnPlateau(opt1, 'min', factor=0.5, patience=3)
    best_f1, wait = -1.0, 0
    stats1 = {}
    best1_path = os.path.join(s1_dir, "best.pt")

    print(f"\n=== {name} Stage1 (Liver) ===")
    for epoch in range(1, num_epochs+1):
        t0 = time.time()
        # — train —
        net1.train()
        total_loss = 0.0
        for imgs, masks in train1:
            imgs, masks = imgs.to(device), masks.to(device).long()
            outs = net1(imgs)
            loss = crit1(outs, masks)
            opt1.zero_grad(); loss.backward(); opt1.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train1)
        sch1.step(avg_loss)

        # — validate & metrics —
        f1_raw, f1_pp = eval_f1s(val1, net1, crit1)
        mean_f1 = 0.5*(f1_pp[1] + f1_pp[2])

        # — checkpoint / early-stop —
        if mean_f1 > best_f1 + 1e-4:
            best_f1, wait = mean_f1, 0
            torch.save(net1.state_dict(), best1_path)
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping S1 at epoch {epoch}")
                break

        dt = time.time() - t0
        print(f"[S1 {epoch:02}/{num_epochs}] "
              f"loss {avg_loss:.4f}  "
              f"F1_liv {f1_pp[1]:.3f}  "
              f"F1_tum {f1_pp[2]:.3f}  "
              f"(best {best_f1:.3f})  {dt:.1f}s")

    stats1 = {"raw_f1": f1_raw, "pp_f1": f1_pp, "best_mean_f1": best_f1}

    # ── STAGE 2 ────────────────────────────────────────────────────────────
    train2 = DataLoader(
        TumourDataset(df_train, width, level, transform=stage2_transform),
        batch_size=bs, shuffle=True,  num_workers=wk, pin_memory=False
    )
    val2   = DataLoader(
        TumourDataset(df_val,   width, level, transform=stage2_val_transform),
        batch_size=bs, shuffle=False, num_workers=wk, pin_memory=False
    )

    net2    = model_cls(1,2).to(device)
    opt2    = Adam(net2.parameters(), lr=1e-4)
    sch2    = ReduceLROnPlateau(opt2, 'min', factor=0.5, patience=3)
    best_f1, wait = -1.0, 0
    stats2 = {}
    best2_path = os.path.join(s2_dir, "best.pt")

    print(f"\n=== {name} Stage2 (Tumour) ===")
    for epoch in range(1, num_epochs+1):
        t0 = time.time()
        net2.train()
        total_loss = 0.0
        for imgs, masks in train2:
            imgs, masks = imgs.to(device), masks.to(device).long()
            outs = net2(imgs)
            loss = crit2(outs, masks)
            opt2.zero_grad(); loss.backward(); opt2.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train2)
        sch2.step(avg_loss)

        f1_raw, f1_pp = eval_f1s(val2, net2, crit2)
        mean_f1 = 0.5*(f1_pp[1] + f1_pp[2])

        if mean_f1 > best_f1 + 1e-4:
            best_f1, wait = mean_f1, 0
            torch.save(net2.state_dict(), best2_path)
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping S2 at epoch {epoch}")
                break

        dt = time.time() - t0
        print(f"[S2 {epoch:02}/{num_epochs}] "
              f"loss {avg_loss:.4f}  "
              f"F1_liv {f1_pp[1]:.3f}  "
              f"F1_tum {f1_pp[2]:.3f}  "
              f"(best {best_f1:.3f})  {dt:.1f}s")

    stats2 = {"raw_f1": f1_raw, "pp_f1": f1_pp, "best_mean_f1": best_f1}

    # return both stats dicts plus both checkpoint paths
    return best1_path, best2_path

In [ ]:
best1_u, best2_u = train_cascade(UNet,          "UNet")
best1_p, best2_p = train_cascade(UNetPlusPlus, "UNet++")

evaluate_cascade_test(UNet,          "UNet",   best1_u, best2_u)
evaluate_cascade_test(UNetPlusPlus, "UNet++", best1_p, best2_p)

pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1



=== UNet Stage1 (Liver) ===


pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1


[S1 01/10] loss 0.2939  F1_liv 0.822  F1_tum 0.000  (best 0.411)  235.5s
[S1 02/10] loss 0.1903  F1_liv 0.871  F1_tum 0.000  (best 0.435)  188.0s
[S1 03/10] loss 0.1372  F1_liv 0.908  F1_tum 0.000  (best 0.454)  187.4s
[S1 04/10] loss 0.1035  F1_liv 0.886  F1_tum 0.000  (best 0.454)  186.9s
[S1 05/10] loss 0.0859  F1_liv 0.891  F1_tum 0.000  (best 0.454)  187.8s
[S1 06/10] loss 0.0753  F1_liv 0.903  F1_tum 0.000  (best 0.454)  217.5s
[S1 07/10] loss 0.0664  F1_liv 0.922  F1_tum 0.000  (best 0.461)  218.0s
[S1 08/10] loss 0.0638  F1_liv 0.933  F1_tum 0.000  (best 0.467)  187.5s
[S1 09/10] loss 0.0575  F1_liv 0.932  F1_tum 0.000  (best 0.467)  186.7s
[S1 10/10] loss 0.0543  F1_liv 0.933  F1_tum 0.000  (best 0.467)  187.5s


pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1



=== UNet Stage2 (Tumour) ===


pixdim[0] (qfac) should be 1 (default) or -1; setting qfac to 1
